## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | 03 - Paired-View YOLO Adaptation Training |
| Model / workflow | DenseNet-121/201 |
| Input | 224x224 |
| Loss | Cross-Entropy (CE) |
| Training / pipeline | Paired-view adaptation/evaluation |
| Result | "train_loss": 0.839801063852023,; "published_qwk": 0.7574319860669029,; "published_macro_f1": 0.6130329903802745, |


# 03 - Paired-View YOLO Adaptation Training

Run notebook 01 first. This applies the selected paired-view idea from `dense_net_121_paired_view_yolo_adaptation.ipynb` in a single training run:

- published 224x224 crop: used 50% of the time
- expanded 1.15x YOLO square ROI: used 50% of the time
- five fine-tuning epochs from a CE DenseNet121 checkpoint
- CE is the only training loss; there is no MSE loss in the source paired-view experiment

Validation reports both published-crop and YOLO-ROI performance, then selects a checkpoint by their mean selection score. Test data is not used to tune this model.


In [1]:
!pip -q install "timm>=1.0" "h5py>=3.9"


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import json
import random
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, cohen_kappa_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Fixed paired-view configuration

Set `BASE_CHECKPOINT` to the CE DenseNet121 checkpoint you want to adapt. This should be the selected checkpoint from the original model, not a CORN or ordinal checkpoint.


In [3]:
SEED = 42
INPUT_SIZE = 224
BATCH_SIZE = 48
NUM_WORKERS = 2
EPOCHS = 5
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-3
ALTERNATE_VIEW_PROBABILITY = 0.50

PUBLISHED_ROOT = Path("/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224")
ROI_ROOT = Path("/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2")
BASE_CHECKPOINT = Path("/content/drive/MyDrive/Models/densenet121_checkpoints/2026-08-04_00-59-15_419728_UTC_natural_orientation_ce_gradcam/best_model.pth")
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f_UTC")
RUN_DIR = Path("/content/drive/MyDrive/Models/densenet121_paired_view_adaptation") / RUN_TIMESTAMP

for required in (PUBLISHED_ROOT, ROI_ROOT, BASE_CHECKPOINT):
    if not required.exists():
        raise FileNotFoundError(required)
RUN_DIR.mkdir(parents=True, exist_ok=False)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


## Build paired published/YOLO records

Every published image must have the same patient-side file in the generated ROI folder. The test split is deliberately excluded from adaptation and checkpoint selection.


In [4]:
rows = []
for split in ("train", "val"):
    for grade in range(5):
        for published_path in sorted((PUBLISHED_ROOT / split / str(grade)).glob("*.png")):
            roi_path = ROI_ROOT / split / str(grade) / published_path.name
            if not roi_path.is_file():
                raise FileNotFoundError(f"Missing paired ROI: {roi_path}")
            rows.append({
                "split": split,
                "grade": grade,
                "published_path": str(published_path),
                "roi_path": str(roi_path),
            })
frame = pd.DataFrame(rows)
print(frame.groupby(["split", "grade"]).size().unstack(fill_value=0))


grade     0     1     2    3    4
split                            
train  2286  1046  1516  757  173
val     328   153   212  106   27


## Preprocessing, paired dataset, and model

The alternate ROI is selected independently for each training sample. This is the paired-view adaptation mechanism. It is not MSE feature matching.


In [5]:
class OpenCVCLAHE:
    def __call__(self, image_rgb):
        lab = cv2.cvtColor(np.asarray(image_rgb), cv2.COLOR_RGB2LAB)
        lightness, a, b = cv2.split(lab)
        lightness = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8)).apply(lightness)
        return cv2.cvtColor(cv2.merge((lightness, a, b)), cv2.COLOR_LAB2RGB)


class SquarePad:
    def __call__(self, image_rgb):
        image = np.asarray(image_rgb)
        height, width = image.shape[:2]
        side = max(height, width)
        top, left = (side - height) // 2, (side - width) // 2
        return cv2.copyMakeBorder(image, top, side - height - top, left, side - width - left, cv2.BORDER_CONSTANT, value=(0, 0, 0))


train_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.50), transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)), transforms.ToTensor(),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class PairedDataset(Dataset):
    def __init__(self, data, transform, alternate_probability):
        self.data = data.reset_index(drop=True)
        self.transform = transform
        self.alternate_probability = alternate_probability
        self.labels = self.data.grade.astype(int).tolist()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        use_roi = self.alternate_probability > 0 and random.random() < self.alternate_probability
        image = cv2.imread(row.roi_path if use_roi else row.published_path, cv2.IMREAD_COLOR)
        if image is None:
            raise IOError(f"Cannot read paired image at index {index}")
        return self.transform(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)), int(row.grade)


class DenseNet121Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model("densenet121", pretrained=False, num_classes=5, drop_rate=0.20)

    def forward(self, images):
        return self.backbone(images)


## Fine-tune for five epochs with cross-entropy

The paired-view source experiment uses `F.cross_entropy`. Its robustness comes from alternating views and validating on both domains, not from an MSE term.


In [6]:
checkpoint = torch.load(BASE_CHECKPOINT, map_location="cpu", weights_only=False)
if checkpoint.get("loss_type") not in (None, "ce"):
    raise RuntimeError(f"Expected CE checkpoint, got {checkpoint.get('loss_type')}")

model = DenseNet121Model().to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
train_frame = frame[frame.split == "train"].copy()
val_frame = frame[frame.split == "val"].copy()
counts = np.bincount(train_frame.grade.to_numpy(), minlength=5)
weights = (1.0 / counts)[train_frame.grade.to_numpy()]
sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True)
train_loader = DataLoader(PairedDataset(train_frame, train_transform, ALTERNATE_VIEW_PROBABILITY), batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")


def evaluate(data, use_roi):
    loader = DataLoader(PairedDataset(data, val_transform, 1.0 if use_roi else 0.0), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    labels, probabilities = [], []
    model.eval()
    with torch.inference_mode():
        for images, batch_labels in loader:
            probs = F.softmax(model(images.to(DEVICE, non_blocking=True)).float(), dim=1).cpu().numpy()
            labels.extend(batch_labels.numpy())
            probabilities.extend(probs)
    labels, probabilities = np.asarray(labels), np.asarray(probabilities)
    predictions = probabilities.argmax(axis=1)
    _, _, f1, _ = precision_recall_fscore_support(labels, predictions, average="macro", zero_division=0)
    ap = average_precision_score(np.eye(5)[labels], probabilities, average="macro")
    qwk = cohen_kappa_score(labels, predictions, weights="quadratic")
    return {"qwk": float(qwk), "macro_f1": float(f1), "macro_ap": float(ap), "selection": float(0.55 * qwk + 0.30 * f1 + 0.15 * ap)}


best_score = -float("inf")
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum, samples = 0.0, 0
    for images, labels in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            loss = F.cross_entropy(model(images), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * len(labels)
        samples += len(labels)
    scheduler.step()
    published = evaluate(val_frame, use_roi=False)
    roi = evaluate(val_frame, use_roi=True)
    robust = 0.5 * (published["selection"] + roi["selection"])
    row = {"epoch": epoch, "train_loss": loss_sum / samples, "robust_selection": robust, **{f"published_{k}": v for k, v in published.items()}, **{f"roi_{k}": v for k, v in roi.items()}}
    history.append(row)
    print(json.dumps(row, indent=2))
    if robust > best_score:
        best_score = robust
        torch.save({"model_state_dict": model.state_dict(), "architecture": "timm_densenet121_linear_gradcam", "loss_type": "ce", "epoch": epoch, "paired_view_probability": ALTERNATE_VIEW_PROBABILITY, "roi_expansion": 1.15, "robust_selection": robust}, RUN_DIR / "best_model.pth")

pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)
(RUN_DIR / "run_config.json").write_text(json.dumps({"loss": "cross_entropy", "mse_used": False, "alternate_view_probability": ALTERNATE_VIEW_PROBABILITY, "roi_expansion": 1.15, "epochs": EPOCHS, "base_checkpoint": str(BASE_CHECKPOINT)}, indent=2))
print("Best checkpoint:", RUN_DIR / "best_model.pth")


epoch 1/5:   0%|          | 0/121 [00:00<?, ?it/s]

{
  "epoch": 1,
  "train_loss": 0.839801063852023,
  "robust_selection": 0.6808177845798176,
  "published_qwk": 0.7574319860669029,
  "published_macro_f1": 0.6130329903802745,
  "published_macro_ap": 0.6957595757176162,
  "published_selection": 0.7048614258085214,
  "roi_qwk": 0.7080774404866351,
  "roi_macro_f1": 0.5834098814101603,
  "roi_macro_ap": 0.6153905777361098,
  "roi_selection": 0.6567741433511138
}


epoch 2/5:   0%|          | 0/121 [00:00<?, ?it/s]

{
  "epoch": 2,
  "train_loss": 0.7839485133300813,
  "robust_selection": 0.6930704940183905,
  "published_qwk": 0.7728370595156995,
  "published_macro_f1": 0.6461884961884963,
  "published_macro_ap": 0.7040595542649835,
  "published_selection": 0.7245258647299311,
  "roi_qwk": 0.7109955872574503,
  "roi_macro_f1": 0.5884537328511047,
  "roi_macro_ap": 0.6268762030661387,
  "roi_selection": 0.6616151233068499
}


epoch 3/5:   0%|          | 0/121 [00:00<?, ?it/s]

{
  "epoch": 3,
  "train_loss": 0.7475351668344975,
  "robust_selection": 0.6931272669565023,
  "published_qwk": 0.7756056098597535,
  "published_macro_f1": 0.6352191554541784,
  "published_macro_ap": 0.6999683636802752,
  "published_selection": 0.7221440866111593,
  "roi_qwk": 0.7152822938220514,
  "roi_macro_f1": 0.5877793405497792,
  "roi_macro_ap": 0.629142556898554,
  "roi_selection": 0.6641104473018452
}


epoch 4/5:   0%|          | 0/121 [00:00<?, ?it/s]

{
  "epoch": 4,
  "train_loss": 0.7372227066650073,
  "robust_selection": 0.6988050836995371,
  "published_qwk": 0.7778117546327442,
  "published_macro_f1": 0.6514403627257649,
  "published_macro_ap": 0.7062716323856062,
  "published_selection": 0.7291693187235797,
  "roi_qwk": 0.7190775681341719,
  "roi_macro_f1": 0.5950677135494528,
  "roi_macro_ap": 0.6295191475790938,
  "roi_selection": 0.6684408486754945
}


epoch 5/5:   0%|          | 0/121 [00:00<?, ?it/s]

{
  "epoch": 5,
  "train_loss": 0.7600074147385848,
  "robust_selection": 0.6941381466068348,
  "published_qwk": 0.7688128896845484,
  "published_macro_f1": 0.6454928796236465,
  "published_macro_ap": 0.705547997858315,
  "published_selection": 0.7223271528923428,
  "roi_qwk": 0.7160957866020006,
  "roi_macro_f1": 0.5916989537949232,
  "roi_macro_ap": 0.6305784770116645,
  "roi_selection": 0.6659491403213269
}
Best checkpoint: /content/drive/MyDrive/Models/densenet121_paired_view_adaptation/2026-08-04_05-11-40_046339_UTC/best_model.pth
